In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))          # notebooks/ -> repo root
from config import directories, logger

import pandas as pd
from sqlalchemy import create_engine, text

DB_PATH = directories.INTERIM_DATA / "panchayat_1.duckdb"
engine = create_engine(f"duckdb:///{DB_PATH}")

def q(sql):
    return pd.read_sql(sql, engine)

print(q("SHOW TABLES").to_string(index=False))

                      name
            activity_asset
activity_community_service
       activity_delegation
      activity_expenditure
             activity_fund
             activity_nsap
         activity_training
          activity_voucher
                  dim_code
            dim_lsdg_theme
        dim_welfare_scheme
            gram_panchayat
                      plan
          planned_activity
                   voucher


In [2]:
#Load tables to add 
SRC = directories.RAW_DATA

df_aa  = pd.read_csv(SRC / "egramswaraj_aa_filtered.csv", low_memory=False)
df_ta  = pd.read_csv(SRC / "egramswaraj_ta_filtered.csv", low_memory=False)
df_aas = pd.read_csv(SRC / "egramswaraj_aa__admapprovalschemewebservice_filtered.csv", low_memory=False)
df_pp  = pd.read_csv(SRC / "egramswaraj_pp__physicalprogressassetstageuploadwebservice_filtered.csv", low_memory=False)
df_re  = pd.read_csv(SRC / "egramswaraj_re__budgetaryallocationschemewebservice_filtered.csv", low_memory=False)

for name, df in [("aa  (admin approval)", df_aa),
                 ("ta  (technical approval)", df_ta),
                 ("aas (approval funding)", df_aas),
                 ("pp  (physical progress)", df_pp),
                 ("re  (budget allocation)", df_re)]:
    print(f"{name:28} {df.shape}")

aa  (admin approval)         (2101, 12)
ta  (technical approval)     (2134, 12)
aas (approval funding)       (2107, 10)
pp  (physical progress)      (8267, 8)
re  (budget allocation)      (2977, 9)


In [3]:
# ============================================================
# BLOCK 1 — admin_approval + admin_approval_scheme
# Unblocks the 14 "Sanctions & Approvals" questions.
# Parent: one row per activity (2,101). Child: funding split by scheme (2,107).
# Idempotent: drops and recreates both tables.
# ============================================================
import pandas as pd
from sqlalchemy import text

# ---------- helper: codes -> clean string (same convention as the rest of the DB) ----------
def to_code(s):
    return (s.astype("string").str.strip()
             .str.replace(r"\.0$", "", regex=True))

# ---------- shape the parent ----------
aa = df_aa.rename(columns={
    "lgd_code":              "gp_lgd_code",
    "gram_panchayat_name":   "gp_name",
    "activityCd":            "activity_code",
    "wrkPlnYr":              "work_plan_year",
    "wrkAdmApprNo":          "adm_approval_no",
    "wrkAdmApprSnctnOrdrDt": "adm_approval_sanction_date",
    "wrkProposedCost":       "work_proposed_cost",
    "wrkAdmApprIssAuthrty":  "adm_approval_authority",
})[["row_id", "gp_lgd_code", "gp_name", "plan_year", "doc_type", "source_file",
    "activity_code", "work_plan_year", "adm_approval_no",
    "adm_approval_sanction_date", "work_proposed_cost",
    "adm_approval_authority"]].copy()

for c in ["row_id", "gp_lgd_code", "activity_code", "adm_approval_no"]:
    aa[c] = to_code(aa[c])
aa["adm_approval_sanction_date"] = pd.to_datetime(
    aa["adm_approval_sanction_date"], errors="coerce", dayfirst=True)

# ---------- shape the child ----------
aas = df_aas.rename(columns={
    "activityCd":            "activity_code",
    "wrkSchmCd":             "scheme_code",
    "wrkSchmCmpntCd":        "scheme_component_code",
    "wrkAdmApprFndSnctnGen": "fund_sanctioned_general",
    "wrkAdmApprFndSnctnSc":  "fund_sanctioned_sc",
    "wrkAdmApprFndSnctnSt":  "fund_sanctioned_st",
    "fndAllctnSchmTot":      "fund_sanctioned_total",
})[["row_id", "parent_row_id", "pos", "activity_code", "scheme_code",
    "scheme_component_code", "fund_sanctioned_general", "fund_sanctioned_sc",
    "fund_sanctioned_st", "fund_sanctioned_total"]].copy()

for c in ["row_id", "parent_row_id", "activity_code",
          "scheme_code", "scheme_component_code"]:
    aas[c] = to_code(aas[c])

# ---------- create tables (parent first, keys built in) ----------
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS admin_approval_scheme"))
    conn.execute(text("DROP TABLE IF EXISTS admin_approval"))
    conn.execute(text("""
        CREATE TABLE admin_approval (
            row_id                     VARCHAR PRIMARY KEY,
            gp_lgd_code                VARCHAR,
            gp_name                    VARCHAR,
            plan_year                  VARCHAR,
            doc_type                   VARCHAR,
            source_file                VARCHAR,
            activity_code              VARCHAR,
            work_plan_year             VARCHAR,
            adm_approval_no            VARCHAR,
            adm_approval_sanction_date TIMESTAMP,
            work_proposed_cost         DECIMAL(16,2),
            adm_approval_authority     VARCHAR,
            FOREIGN KEY (activity_code) REFERENCES planned_activity (activity_code),
            FOREIGN KEY (gp_lgd_code)   REFERENCES gram_panchayat (gp_lgd_code)
        )"""))
    conn.execute(text("""
        CREATE TABLE admin_approval_scheme (
            row_id                  VARCHAR PRIMARY KEY,
            parent_row_id           VARCHAR,
            pos                     BIGINT,
            activity_code           VARCHAR,
            scheme_code             VARCHAR,
            scheme_component_code   VARCHAR,
            fund_sanctioned_general DECIMAL(16,2),
            fund_sanctioned_sc      DECIMAL(16,2),
            fund_sanctioned_st      DECIMAL(16,2),
            fund_sanctioned_total   DECIMAL(16,2),
            FOREIGN KEY (parent_row_id) REFERENCES admin_approval (row_id)
        )"""))

# ---------- load parent then child ----------
aa.to_sql("admin_approval", engine, if_exists="append", index=False)
aas.to_sql("admin_approval_scheme", engine, if_exists="append", index=False)

# ---------- verify ----------
print("admin_approval        :", q("SELECT count(*) FROM admin_approval").iloc[0, 0])
print("admin_approval_scheme :", q("SELECT count(*) FROM admin_approval_scheme").iloc[0, 0])
print("\nsample:")
display(q("""
    SELECT a.activity_code, a.adm_approval_no, a.adm_approval_sanction_date,
           a.work_proposed_cost, a.adm_approval_authority,
           s.fund_sanctioned_total
    FROM admin_approval a
    LEFT JOIN admin_approval_scheme s ON s.parent_row_id = a.row_id
    LIMIT 5
"""))

admin_approval        : 2101
admin_approval_scheme : 2107

sample:


,activity_code,adm_approval_no,adm_approval_sanction_date,work_proposed_cost,adm_approval_authority,fund_sanctioned_total
0,44434659,01,2021-10-02,48000.0,BDO,48000.0
1,44434533,03,2021-10-02,180000.0,BDO,180000.0
2,44434332,02,2021-10-02,70000.0,BDO,70000.0
3,50379473,04,NaT,54492.0,BDO,54492.0
4,50380045,02,NaT,33002.0,BDO,33002.0


In [4]:
q("""
SELECT adm_approval_no, count(*) AS n
FROM admin_approval
GROUP BY 1 ORDER BY n DESC LIMIT 10
""")

,adm_approval_no,n
0,1,324
1,01,154
2,2,116
3,4,90
4,5,86
5,3,82
6,6,57
7,18,51
8,12,49
9,02,45


In [5]:
q("""
SELECT adm_approval_no,
       CASE WHEN adm_approval_no LIKE '0%' THEN 'padded' ELSE 'plain' END AS form,
       count(*) AS n
FROM admin_approval
WHERE adm_approval_no IN ('1','01','2','02','3','03','4','04','5','05')
GROUP BY 1,2 ORDER BY adm_approval_no
""")

,adm_approval_no,form,n
0,01,padded,154
1,02,padded,45
2,03,padded,23
3,04,padded,18
4,05,padded,19
5,1,plain,324
6,2,plain,116
7,3,plain,82
8,4,plain,90
9,5,plain,86


In [7]:
#date check
q("""SELECT count(adm_approval_sanction_date) AS has_date,
            count(*) - count(adm_approval_sanction_date) AS missing,
            min(adm_approval_sanction_date) AS earliest,
            max(adm_approval_sanction_date) AS latest
     FROM admin_approval""")
print(df_aa["wrkAdmApprSnctnOrdrDt"].dropna().head(15).tolist())

['2021-02-10', '2021-02-10', '2021-02-10', '2021-02-25', '2021-02-25', '2021-02-25', '2021-04-12', '2020-11-06', '2021-02-10', '2021-02-10', '2021-02-10', '2021-04-01', '2020-11-06', '2020-11-06', '2021-02-26']


In [8]:
# 2. are the 987 missing dates genuinely blank in the source?
print("blank in source :", df_aa["wrkAdmApprSnctnOrdrDt"].isna().sum())
print("NaT in database :", q("SELECT count(*) FROM admin_approval WHERE adm_approval_sanction_date IS NULL").iloc[0,0])

blank in source : 0
NaT in database : 987


In [9]:
# which raw values failed to parse?
parsed = pd.to_datetime(df_aa["wrkAdmApprSnctnOrdrDt"], errors="coerce", dayfirst=True)
failed = df_aa.loc[parsed.isna(), "wrkAdmApprSnctnOrdrDt"]

print("failed to parse:", len(failed))
print("\ndistinct forms (first 20):")
print(failed.astype(str).unique()[:20])
print("\nstring lengths:")
print(failed.astype(str).str.len().value_counts().to_string())

failed to parse: 987

distinct forms (first 20):
<ArrowStringArray>
['2021-02-25', '2021-02-26', '2021-12-22', '2021-12-23', '2021-12-24',
 '2021-12-18', '2025-09-29', '2022-07-22', '2022-12-28', '2023-02-14',
 '2023-02-23', '2023-06-19', '2023-12-26', '2024-12-24', '2025-05-23',
 '2024-12-22', '2025-07-28', '2025-06-18', '2025-07-31', '2025-11-20']
Length: 20, dtype: str

string lengths:
wrkAdmApprSnctnOrdrDt
10    987


In [10]:
#corrected block 1
# ============================================================
# BLOCK 1 — admin_approval + admin_approval_scheme
# Unblocks the 14 "Sanctions & Approvals" questions.
# Parent: one row per activity (2,101). Child: funding by scheme (2,107).
# Dates are ISO (YYYY-MM-DD) — no dayfirst.
# Approval numbers are sequence numbers, normalised to strip leading zeros.
# Idempotent: drops and recreates both tables.
# ============================================================
import pandas as pd
from sqlalchemy import text

# ---------- helper: codes -> clean string (same convention as the rest of the DB) ----------
def to_code(s):
    return (s.astype("string").str.strip()
             .str.replace(r"\.0$", "", regex=True))

# ---------- shape the parent ----------
aa = df_aa.rename(columns={
    "lgd_code":              "gp_lgd_code",
    "gram_panchayat_name":   "gp_name",
    "activityCd":            "activity_code",
    "wrkPlnYr":              "work_plan_year",
    "wrkAdmApprNo":          "adm_approval_no",
    "wrkAdmApprSnctnOrdrDt": "adm_approval_sanction_date",
    "wrkProposedCost":       "work_proposed_cost",
    "wrkAdmApprIssAuthrty":  "adm_approval_authority",
})[["row_id", "gp_lgd_code", "gp_name", "plan_year", "doc_type", "source_file",
    "activity_code", "work_plan_year", "adm_approval_no",
    "adm_approval_sanction_date", "work_proposed_cost",
    "adm_approval_authority"]].copy()

for c in ["row_id", "gp_lgd_code", "activity_code", "adm_approval_no"]:
    aa[c] = to_code(aa[c])

# ISO dates — dayfirst=True would fail on all of them
aa["adm_approval_sanction_date"] = pd.to_datetime(
    aa["adm_approval_sanction_date"], errors="coerce", format="ISO8601")

# strip leading zeros: '01' and '1' are the same sequence number
aa["adm_approval_no"] = aa["adm_approval_no"].str.lstrip("0").replace("", "0")

# ---- guard: did anything fail to parse that wasn't blank in the source? ----
src_blank = df_aa["wrkAdmApprSnctnOrdrDt"].isna().sum()
parsed_na = aa["adm_approval_sanction_date"].isna().sum()
assert parsed_na == src_blank, (
    f"date parse lost {parsed_na - src_blank} rows "
    f"(source blanks={src_blank}, nulls after parse={parsed_na})")

# ---------- shape the child ----------
aas = df_aas.rename(columns={
    "activityCd":            "activity_code",
    "wrkSchmCd":             "scheme_code",
    "wrkSchmCmpntCd":        "scheme_component_code",
    "wrkAdmApprFndSnctnGen": "fund_sanctioned_general",
    "wrkAdmApprFndSnctnSc":  "fund_sanctioned_sc",
    "wrkAdmApprFndSnctnSt":  "fund_sanctioned_st",
    "fndAllctnSchmTot":      "fund_sanctioned_total",
})[["row_id", "parent_row_id", "pos", "activity_code", "scheme_code",
    "scheme_component_code", "fund_sanctioned_general", "fund_sanctioned_sc",
    "fund_sanctioned_st", "fund_sanctioned_total"]].copy()

for c in ["row_id", "parent_row_id", "activity_code",
          "scheme_code", "scheme_component_code"]:
    aas[c] = to_code(aas[c])

# ---------- create tables (parent first, keys built in) ----------
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS admin_approval_scheme"))
    conn.execute(text("DROP TABLE IF EXISTS admin_approval"))
    conn.execute(text("""
        CREATE TABLE admin_approval (
            row_id                     VARCHAR PRIMARY KEY,
            gp_lgd_code                VARCHAR,
            gp_name                    VARCHAR,
            plan_year                  VARCHAR,
            doc_type                   VARCHAR,
            source_file                VARCHAR,
            activity_code              VARCHAR,
            work_plan_year             VARCHAR,
            adm_approval_no            VARCHAR,
            adm_approval_sanction_date TIMESTAMP,
            work_proposed_cost         DECIMAL(16,2),
            adm_approval_authority     VARCHAR,
            FOREIGN KEY (activity_code) REFERENCES planned_activity (activity_code),
            FOREIGN KEY (gp_lgd_code)   REFERENCES gram_panchayat (gp_lgd_code)
        )"""))
    conn.execute(text("""
        CREATE TABLE admin_approval_scheme (
            row_id                  VARCHAR PRIMARY KEY,
            parent_row_id           VARCHAR,
            pos                     BIGINT,
            activity_code           VARCHAR,
            scheme_code             VARCHAR,
            scheme_component_code   VARCHAR,
            fund_sanctioned_general DECIMAL(16,2),
            fund_sanctioned_sc      DECIMAL(16,2),
            fund_sanctioned_st      DECIMAL(16,2),
            fund_sanctioned_total   DECIMAL(16,2),
            FOREIGN KEY (parent_row_id) REFERENCES admin_approval (row_id)
        )"""))

# ---------- load parent then child ----------
aa.to_sql("admin_approval", engine, if_exists="append", index=False)
aas.to_sql("admin_approval_scheme", engine, if_exists="append", index=False)

# ---------- verify ----------
print("admin_approval        :", q("SELECT count(*) FROM admin_approval").iloc[0, 0], "(expect 2101)")
print("admin_approval_scheme :", q("SELECT count(*) FROM admin_approval_scheme").iloc[0, 0], "(expect 2107)")
print()
display(q("""
    SELECT count(*) AS n_rows,
           count(adm_approval_sanction_date) AS has_date,
           min(adm_approval_sanction_date)   AS earliest,
           max(adm_approval_sanction_date)   AS latest
    FROM admin_approval
"""))
display(q("""
    SELECT a.activity_code, a.adm_approval_no, a.adm_approval_sanction_date,
           a.work_proposed_cost, a.adm_approval_authority, s.fund_sanctioned_total
    FROM admin_approval a
    LEFT JOIN admin_approval_scheme s ON s.parent_row_id = a.row_id
    LIMIT 5
"""))

admin_approval        : 2101 (expect 2101)
admin_approval_scheme : 2107 (expect 2107)



,n_rows,has_date,earliest,latest
0,2101,2101,2019-10-02,2026-08-19


,activity_code,adm_approval_no,adm_approval_sanction_date,work_proposed_cost,adm_approval_authority,fund_sanctioned_total
0,44434659,1,2021-02-10,48000.0,BDO,48000.0
1,44434533,3,2021-02-10,180000.0,BDO,180000.0
2,44434332,2,2021-02-10,70000.0,BDO,70000.0
3,50379473,4,2021-02-25,54492.0,BDO,54492.0
4,50380045,2,2021-02-25,33002.0,BDO,33002.0


In [11]:
# ============================================================
# BLOCK 2 — technical_approval
# One row per activity (2,134). Standalone: no child table.
# Verifies date format before parsing, and guards against silent parse loss.
# Idempotent.
# ============================================================
import pandas as pd
from sqlalchemy import text

# ---- inspect the date format before deciding how to parse ----
print("wrkTecApprOrdrDt sample:", df_ta["wrkTecApprOrdrDt"].dropna().head(5).tolist())
print("blank in source        :", df_ta["wrkTecApprOrdrDt"].isna().sum(), "of", len(df_ta))

# ---------- shape ----------
ta = df_ta.rename(columns={
    "lgd_code":             "gp_lgd_code",
    "gram_panchayat_name":  "gp_name",
    "activityCd":           "activity_code",
    "wrkTecApprReqFlg":     "tec_approval_required",
    "wrkTecApprCost":       "tec_approval_cost",
    "wrkTecApprIssAuthrty": "tec_approval_authority",
    "wrkTecApprOrdrNo":     "tec_approval_order_no",
    "wrkTecApprOrdrDt":     "tec_approval_order_date",
})[["row_id", "gp_lgd_code", "gp_name", "plan_year", "doc_type", "source_file",
    "activity_code", "tec_approval_required", "tec_approval_cost",
    "tec_approval_authority", "tec_approval_order_no",
    "tec_approval_order_date"]].copy()

for c in ["row_id", "gp_lgd_code", "activity_code", "tec_approval_order_no"]:
    ta[c] = to_code(ta[c])

# ISO dates, same as admin approval
ta["tec_approval_order_date"] = pd.to_datetime(
    ta["tec_approval_order_date"], errors="coerce", format="ISO8601")

# order numbers are sequence numbers -> strip leading zeros
ta["tec_approval_order_no"] = ta["tec_approval_order_no"].str.lstrip("0").replace("", "0")

# ---- guard: no silent parse loss ----
src_blank = df_ta["wrkTecApprOrdrDt"].isna().sum()
parsed_na = ta["tec_approval_order_date"].isna().sum()
assert parsed_na == src_blank, (
    f"date parse lost {parsed_na - src_blank} rows "
    f"(source blanks={src_blank}, nulls after parse={parsed_na})")

# ---------- create ----------
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS technical_approval"))
    conn.execute(text("""
        CREATE TABLE technical_approval (
            row_id                  VARCHAR PRIMARY KEY,
            gp_lgd_code             VARCHAR,
            gp_name                 VARCHAR,
            plan_year               VARCHAR,
            doc_type                VARCHAR,
            source_file             VARCHAR,
            activity_code           VARCHAR,
            tec_approval_required   VARCHAR,
            tec_approval_cost       DECIMAL(16,2),
            tec_approval_authority  VARCHAR,
            tec_approval_order_no   VARCHAR,
            tec_approval_order_date TIMESTAMP,
            FOREIGN KEY (activity_code) REFERENCES planned_activity (activity_code),
            FOREIGN KEY (gp_lgd_code)   REFERENCES gram_panchayat (gp_lgd_code)
        )"""))

ta.to_sql("technical_approval", engine, if_exists="append", index=False)

# ---------- verify ----------
print("\ntechnical_approval:", q("SELECT count(*) FROM technical_approval").iloc[0, 0], "(expect 2134)")
display(q("""
    SELECT count(*) AS n_rows,
           count(tec_approval_order_date) AS has_date,
           min(tec_approval_order_date) AS earliest,
           max(tec_approval_order_date) AS latest
    FROM technical_approval
"""))
display(q("SELECT * FROM technical_approval LIMIT 5"))

wrkTecApprOrdrDt sample: ['2021-03-27', '2021-03-27', '2021-03-27', '2021-02-25', '2021-02-25']
blank in source        : 0 of 2134

technical_approval: 2134 (expect 2134)


,n_rows,has_date,earliest,latest
0,2134,2134,2019-03-13,2026-07-09


,row_id,gp_lgd_code,gp_name,plan_year,doc_type,source_file,activity_code,tec_approval_required,tec_approval_cost,tec_approval_authority,tec_approval_order_no,tec_approval_order_date
0,116350|2020|TA|0,116350,Hirlipali,2020,TA,2020_TA.json,44434659,N,NaN,NR,NR,2021-03-27
1,116350|2020|TA|1,116350,Hirlipali,2020,TA,2020_TA.json,44434332,N,NaN,NR,NR,2021-03-27
2,116350|2020|TA|2,116350,Hirlipali,2020,TA,2020_TA.json,44434533,N,NaN,NR,NR,2021-03-27
3,116350|2020|TA|3,116350,Hirlipali,2020,TA,2020_TA.json,50380045,R,33002.0,BDO,2,2021-02-25
4,116350|2020|TA|4,116350,Hirlipali,2020,TA,2020_TA.json,50379473,R,54492.0,BDO,4,2021-02-25


In [12]:
#check techincal approval 
q("""
SELECT tec_approval_required,
       count(*) AS n,
       count(tec_approval_cost) AS has_cost,
       sum(CASE WHEN tec_approval_authority = 'NR' THEN 1 ELSE 0 END) AS authority_is_NR,
       sum(CASE WHEN tec_approval_order_no  = 'NR' THEN 1 ELSE 0 END) AS order_no_is_NR
FROM technical_approval
GROUP BY 1 ORDER BY n DESC
""")

,tec_approval_required,n,has_cost,authority_is_NR,order_no_is_NR
0,R,2060,2060,0,0
1,N,74,0,74,74


In [15]:
#check coordinated 
lat_num = pd.to_numeric(df_pp["latitude"], errors="coerce")
failed = df_pp.loc[lat_num.isna(), ["latitude", "longitude"]]
print("failed:", len(failed))
print(failed["latitude"].astype(str).unique()[:15])
print("\nlengths:", failed["latitude"].astype(str).str.len().value_counts().to_string())

failed: 689
<ArrowStringArray>
[                '21.3711999,21.3712009',
                 '21.3711622,21.3711618',
 '21.371296666666662,21.371299999999998',
               '21.37111851,21.37112698',
               '21.37116265,21.37116161',
                   '21.3711687,21.37117',
                 '21.3712152,21.3711218',
                 '21.3711832,21.3711891',
                 '21.3711882,21.3711882',
                  '21.3711576,21.371156',
               '21.41173979,21.41173019',
               '21.41176542,21.41175831',
                '21.41175009,21.4117482',
               '21.41174907,21.41176483',
                '21.3697001,21.36969824']
Length: 15, dtype: str

lengths: latitude
21    175
23    126
37     89
22     50
36     40
35     38
28     24
19     20
20     16
32     16
34     15
27     15
29     14
17      6
47      6
18      5
75      5
56      4
46      3
55      3
45      2
31      2
33      2
53      2
59      2
54      1
73      1
16      1
63      1
43     

In [16]:
lon_num = pd.to_numeric(df_pp["longitude"], errors="coerce")
lon_failed = df_pp.loc[lon_num.isna(), "longitude"]
print("lon failed:", len(lon_failed))
print(lon_failed.astype(str).unique()[:8])

# do lat and lon fail on the same rows?
lat_num = pd.to_numeric(df_pp["latitude"], errors="coerce")
print("\nsame rows fail for both?",
      (lat_num.isna() == lon_num.isna()).all())

# how many values per cell?
n_parts = df_pp.loc[lat_num.isna(), "latitude"].astype(str).str.count(",") + 1
print("\nvalues per cell:")
print(n_parts.value_counts().to_string())

lon failed: 689
<ArrowStringArray>
[              '83.7815807,83.7815986',               '83.7815992,83.7815982',
 '83.78146999999998,83.78152833333334',             '83.78174866,83.78172683',
             '83.78169221,83.78169213',               '83.7816148,83.7816171',
               '83.7814197,83.7815408',               '83.7815776,83.7815785']
Length: 8, dtype: str

same rows fail for both? True

values per cell:
latitude
2    605
3     67
4     14
5      2
6      1


In [17]:
# ============================================================
# BLOCK 3 — physical_progress  (corrected)
# 8,267 rows across 1,675 activities — one-to-many, row_id is the PK.
# 689 rows hold 2-6 comma-separated GPS captures in one cell; we take the
# first and keep the raw string in *_raw so nothing is discarded.
# Idempotent.
# ============================================================
import pandas as pd
from sqlalchemy import text

def first_coord(s):
    """'21.371,21.372' -> 21.371 ; plain numbers pass through unchanged."""
    return pd.to_numeric(
        s.astype("string").str.split(",").str[0].str.strip(),
        errors="coerce")

# ---------- shape ----------
pp = df_pp.rename(columns={
    "activityCd":     "activity_code",
    "fileUploadId":   "file_upload_id",
    "plnunttypecode": "plan_unit_type_code",
})[["row_id", "parent_row_id", "pos", "activity_code", "file_upload_id",
    "longitude", "latitude", "plan_unit_type_code"]].copy()

for c in ["row_id", "parent_row_id", "activity_code",
          "file_upload_id", "plan_unit_type_code"]:
    pp[c] = to_code(pp[c])

# keep the raw strings, then extract the first coordinate
pp["latitude_raw"]  = df_pp["latitude"].astype("string").values
pp["longitude_raw"] = df_pp["longitude"].astype("string").values
pp["n_coords"]      = (df_pp["latitude"].astype("string")
                         .str.count(",").fillna(0).astype(int).values + 1)
pp["latitude"]      = first_coord(df_pp["latitude"]).values
pp["longitude"]     = first_coord(df_pp["longitude"]).values

# ---- guard: no silent coordinate loss ----
src_blank = df_pp["latitude"].isna().sum()
parsed_na = pp["latitude"].isna().sum()
assert parsed_na == src_blank, (
    f"coord parse lost {parsed_na - src_blank} rows "
    f"(source blanks={src_blank}, nulls after parse={parsed_na})")

pp = pp[["row_id", "parent_row_id", "pos", "activity_code", "file_upload_id",
         "longitude", "latitude", "n_coords", "longitude_raw", "latitude_raw",
         "plan_unit_type_code"]]

# ---------- create ----------
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS physical_progress"))
    conn.execute(text("""
        CREATE TABLE physical_progress (
            row_id              VARCHAR PRIMARY KEY,
            parent_row_id       VARCHAR,
            pos                 BIGINT,
            activity_code       VARCHAR,
            file_upload_id      VARCHAR,
            longitude           DOUBLE,
            latitude            DOUBLE,
            n_coords            INTEGER,
            longitude_raw       VARCHAR,
            latitude_raw        VARCHAR,
            plan_unit_type_code VARCHAR,
            FOREIGN KEY (activity_code) REFERENCES planned_activity (activity_code)
        )"""))

pp.to_sql("physical_progress", engine, if_exists="append", index=False)

# ---------- verify ----------
print("physical_progress:", q("SELECT count(*) FROM physical_progress").iloc[0, 0], "(expect 8267)")
display(q("""
    SELECT count(*) AS n_rows,
           count(DISTINCT activity_code) AS activities,
           count(latitude) AS has_coords,
           sum(CASE WHEN n_coords > 1 THEN 1 ELSE 0 END) AS multi_capture_rows,
           round(min(latitude),4) AS lat_min, round(max(latitude),4) AS lat_max,
           round(min(longitude),4) AS lon_min, round(max(longitude),4) AS lon_max
    FROM physical_progress
"""))

# ---------- coordinate sanity: Odisha is ~17.8-22.6N, 81.4-87.5E ----------
display(q("""
    SELECT sum(CASE WHEN latitude BETWEEN 17.5 AND 22.8
                     AND longitude BETWEEN 81.2 AND 87.7 THEN 1 ELSE 0 END) AS inside_odisha,
           sum(CASE WHEN latitude NOT BETWEEN 17.5 AND 22.8
                      OR longitude NOT BETWEEN 81.2 AND 87.7 THEN 1 ELSE 0 END) AS outside
    FROM physical_progress WHERE latitude IS NOT NULL
"""))
display(q("""
    SELECT activity_code, latitude, longitude, n_coords, count(*) AS n
    FROM physical_progress
    WHERE latitude IS NOT NULL
      AND (latitude NOT BETWEEN 17.5 AND 22.8 OR longitude NOT BETWEEN 81.2 AND 87.7)
    GROUP BY 1,2,3,4 ORDER BY n DESC LIMIT 15
"""))

physical_progress: 8267 (expect 8267)


,n_rows,activities,has_coords,multi_capture_rows,lat_min,lat_max,lon_min,lon_max
0,8267,1675,8267,689,10.3466,22.3499,81.7093,97.0192


,inside_odisha,outside
0,8266,1


,activity_code,latitude,longitude,n_coords,n
0,124908159,10.346632,97.019223,1,1


In [18]:
#check tables
# all tables and counts
print(q("SHOW TABLES").to_string(index=False))
print()
for t in ["admin_approval", "admin_approval_scheme",
          "technical_approval", "physical_progress"]:
    print(f"{t:24} {q(f'SELECT count(*) FROM {t}').iloc[0,0]}")

# foreign keys resolve? (should all be 0)
print("\norphan check:")
for t in ["admin_approval", "technical_approval", "physical_progress"]:
    n = q(f"""SELECT count(*) FROM {t} c
              LEFT JOIN planned_activity p USING (activity_code)
              WHERE p.activity_code IS NULL""").iloc[0,0]
    print(f"  {t:24} activity_code orphans: {n}")

                      name
            activity_asset
activity_community_service
       activity_delegation
      activity_expenditure
             activity_fund
             activity_nsap
         activity_training
          activity_voucher
            admin_approval
     admin_approval_scheme
                  dim_code
            dim_lsdg_theme
        dim_welfare_scheme
            gram_panchayat
         physical_progress
                      plan
          planned_activity
        technical_approval
                   voucher

admin_approval           2101
admin_approval_scheme    2107
technical_approval       2134
physical_progress        8267

orphan check:
  admin_approval           activity_code orphans: 0
  technical_approval       activity_code orphans: 0
  physical_progress        activity_code orphans: 0


In [19]:
#check 
q("""
SELECT a.activity_code, a.activity_name,
       aa.adm_approval_no, aa.adm_approval_sanction_date,
       aa.work_proposed_cost, aa.adm_approval_authority,
       ta.tec_approval_cost, ta.tec_approval_order_date,
       e.total_expenditure
FROM planned_activity a
JOIN admin_approval aa      USING (activity_code)
LEFT JOIN technical_approval ta USING (activity_code)
LEFT JOIN activity_expenditure e USING (activity_code)
WHERE a.gp_lgd_code = '119598'
ORDER BY aa.adm_approval_sanction_date DESC
LIMIT 10
""")

,activity_code,activity_name,adm_approval_no,adm_approval_sanction_date,work_proposed_cost,adm_approval_authority,tec_approval_cost,tec_approval_order_date,total_expenditure
0,122887323,Construction of roads,27,2026-01-02,150000.0,SARAPANCHA,150000.0,2026-01-02,73572.0
1,127378005,Construction of roads,2,2025-11-30,150000.0,Sarapancha,150000.0,2025-11-30,147384.0
2,127377934,Construction of roads,1,2025-11-30,200000.0,Sarapancha,200000.0,2025-11-30,196480.0
3,65617848,Technical & administrative expenses,10,2025-11-01,8000.0,Sarapancha,8000.0,2023-10-01,5900.0
4,122434551,Operation and Maintenance of drinking/piped wa...,10,2025-05-02,50000.0,SARAPANCHA,50000.0,2025-05-02,0.0
5,122431817,Operation and Maintenance of drinking/piped wa...,3,2025-05-02,160000.0,SARAPANCHA,160000.0,2025-05-02,0.0
6,122408190,Operation & maintenance of community sanitary ...,9,2025-05-02,201672.0,SARAPANCHA,201672.0,2025-05-02,201672.0
7,122403174,Operation and Maintenance of drinking/piped wa...,2,2025-05-02,300000.0,SARAPANCHA,300000.0,2025-05-02,0.0
8,101745219,Drainage Construction,28,2025-05-02,270672.0,SARAPANCHA,270672.0,2025-05-02,266388.0
9,122425822,Purchase of Tricycles/other battery -operated ...,20,2025-05-02,84000.0,SARAPANCHA,84000.0,2025-05-02,0.0


In [20]:
from sqlalchemy import text
with engine.begin() as conn:
    conn.execute(text("CHECKPOINT"))
engine.dispose()
print("flushed and closed")

flushed and closed
